# LeetCode 3209 — Number of Subarrays With AND Value of K bằng Python

- Link: https://leetcode.com/problems/number-of-subarrays-with-and-value-of-k/
- Độ khó: **Hard**
- Chủ đề: **Tính đơn điệu của AND** — với đầu phải cố định, AND chỉ giảm (theo tập bit) khi kéo dài sang trái và chỉ nhận `≤ 31` giá trị khác nhau; các kỹ thuật: tập giá trị AND kết thúc tại `r`, mẹo "ghi đè tại chỗ + bisect", cửa sổ trượt đếm bit với `count(≥k) − count(≥k+1)`

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countSubarrays(self, nums: List[int], k: int) -> int:
```

## 1. Mục tiêu học

1. Với `r` cố định, `f(l) = nums[l] & … & nums[r]` **không giảm** theo `l` (kéo dài sang trái chỉ tắt bớt bit) và có `≤ 31` giá trị khác nhau (mỗi lần đổi giá trị mất ít nhất một bit).
2. Đếm bằng **từ điển {giá trị AND: số đầu trái}** cho các mảng con kết thúc tại `r`: `O(n log V)`.
3. Mẹo LC 1521: ghi đè `nums[i] &= nums[r]` lùi về trái, dừng sớm khi không đổi; khi đó `nums[0..r]` **đã sắp tăng** ⇒ đếm `k` bằng hai lần `bisect`.
4. Cửa sổ trượt: `#(AND = k) = #(AND ≥ k) − #(AND ≥ k + 1)`, vì "AND ≥ t" đóng với việc thu hẹp cửa sổ.
5. Kiểm chứng bằng brute `O(n²)`.

## 2. Đề bài & ví dụ

Cho mảng `nums` và số nguyên `k`. Đếm số **mảng con** (liên tiếp, khác rỗng) có AND bit bằng đúng `k`.

**Ví dụ 1:** `nums = [1,1,1], k = 1` → `6`. **Ví dụ 2:** `nums = [1,1,2], k = 1` → `3`. **Ví dụ 3:** `nums = [1,2,3], k = 2` → `2`.

**Ràng buộc:** `1 ≤ n ≤ 10^5`; `0 ≤ nums[i], k ≤ 10^9`.

In [1]:
from typing import List
from bisect import bisect_left, bisect_right
import random, time

examples = [([1, 1, 1], 1, 6), ([1, 1, 2], 1, 3), ([1, 2, 3], 2, 2)]
for a, k, exp in examples:
    print(a, 'k =', k, '-> kỳ vọng', exp)

[1, 1, 1] k = 1 -> kỳ vọng 6
[1, 1, 2] k = 1 -> kỳ vọng 3
[1, 2, 3] k = 2 -> kỳ vọng 2


## 3. Kiến thức nền

### 3.1. AND chỉ tắt bit

`x & y ⊆ x` (theo tập bit) nên `x & y ≤ x`. Với `r` cố định, dãy `f(r), f(r−1), …, f(0)` (với `f(l) = AND(l..r)`) không tăng, và mỗi lần **giảm** thì mất ít nhất một bit ⇒ tối đa `31` lần giảm (giá trị `≤ 10^9 < 2^30`).

In [2]:
def and_values_ending_at(nums, r):
    vals = []; cur = -1                     # -1 = mọi bit bật
    for l in range(r, -1, -1):
        cur &= nums[l]
        vals.append(cur)
    return vals

rng = random.Random(3209)
a = [rng.randint(0, 2 ** 10 - 1) | 0b1 for _ in range(12)]
v = and_values_ending_at(a, 11)
print('AND(l..11) khi l giảm dần:', v)
print('số giá trị khác nhau:', len(set(v)), '| không tăng:', all(v[i] >= v[i + 1] for i in range(len(v) - 1)))

AND(l..11) khi l giảm dần: [207, 71, 67, 3, 3, 3, 1, 1, 1, 1, 1, 1]
số giá trị khác nhau: 5 | không tăng: True


### 3.2. "AND ≥ t" có tính đóng khi thu hẹp

Nếu cửa sổ `[l, r]` có AND `≥ t` thì mọi cửa sổ con `[l', r']` có AND `⊇` AND cũ nên `≥ t` (AND của tập con lớn hơn hoặc bằng — theo tập bit **và** theo giá trị). ⇒ Với mỗi `r`, các `l` thoả "AND ≥ t" là một **đoạn** `[L_t(r), r]`, và `L_t(r)` không giảm theo `r` ⇒ hai con trỏ.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

Đếm theo đầu phải `r`: cần biết có bao nhiêu `l` cho `AND(l..r) = k`. Vì `AND(l..r)` chỉ có ≤ 31 giá trị, ta mang theo một **bảng nhỏ** `{giá trị: số l}` và cập nhật khi thêm `nums[r]`: mỗi giá trị cũ `v` thành `v & nums[r]`, cộng thêm mảng con một phần tử `nums[r]`.

Ví dụ 2 (`[1, 1, 2]`, `k = 1`): `r = 0`: `{1: 1}` (+1) → `r = 1`: `{1: 2}` (+2) → `r = 2`: `{0: 2, 2: 1}` (+0). Tổng `3`.

## 5. Cách 1 — Brute Force `O(n²)` (dừng sớm khi AND đã nhỏ hơn `k`)

In [3]:
def brute(nums, k):
    n = len(nums); cnt = 0
    for l in range(n):
        cur = -1
        for r in range(l, n):
            cur &= nums[r]
            if cur == k:
                cnt += 1
    return cnt

for a, k, exp in examples:
    print('brute:', brute(a, k), '| kỳ vọng', exp)

brute: 6 | kỳ vọng 6
brute: 3 | kỳ vọng 3
brute: 2 | kỳ vọng 2


Tối ưu nhỏ: khi `cur & k != k` (đã mất một bit của `k`) thì không bao giờ quay lại `k` được ⇒ `break`. Vẫn `O(n²)` xấu nhất (ví dụ mọi phần tử bằng `k`).

## 6. Ý tưởng thuật toán / bước đột phá

```
ans = 0; cur = {}                   # {AND(l..r-1): số l}
for x in nums:
    nxt = {x: 1}
    for v, c in cur.items():
        nxt[v & x] = nxt.get(v & x, 0) + c
    cur = nxt
    ans += cur.get(k, 0)
```

`|cur| ≤ 31` ⇒ `O(n · 31)`.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — Từ điển giá trị AND kết thúc tại `r`

In [4]:
def and_dict(nums, k):
    ans = 0; cur = {}
    for x in nums:
        nxt = {x: 1}
        for v, c in cur.items():
            w = v & x
            nxt[w] = nxt.get(w, 0) + c
        cur = nxt
        ans += cur.get(k, 0)
    return ans

for a, k, exp in examples:
    print('dict:', and_dict(a, k), '| kỳ vọng', exp)

dict: 6 | kỳ vọng 6
dict: 3 | kỳ vọng 3
dict: 2 | kỳ vọng 2


### 7.2. Cách 3 — Ghi đè tại chỗ + `bisect` (mẹo LC 1521)

Duy trì bất biến: sau bước `r`, `b[i] = AND(i..r)` với mọi `i ≤ r`. Khi thêm `x = nums[r]`, đi lùi `i = r−1, r−2, …` và gán `b[i] &= x`; **dừng ngay** khi `b[i] & x == b[i]` (không đổi) — vì `b[i−1] ⊆ b[i] ⊆ x` theo bit nên các vị trí xa hơn cũng không đổi. Mỗi `b[i]` chỉ đổi tối đa 30 lần ⇒ khấu hao `O(n log V)`. Do `b[0..r]` không giảm, số `i` với `b[i] = k` là `bisect_right − bisect_left`.

In [5]:
def inplace_bisect(nums, k):
    b = list(nums); ans = 0
    for r, x in enumerate(b):
        i = r - 1
        while i >= 0 and b[i] & x != b[i]:
            b[i] &= x
            i -= 1
        ans += bisect_right(b, k, 0, r + 1) - bisect_left(b, k, 0, r + 1)
    return ans

for a, k, exp in examples:
    print('in-place + bisect:', inplace_bisect(a, k), '| kỳ vọng', exp)

in-place + bisect: 6 | kỳ vọng 6
in-place + bisect: 3 | kỳ vọng 3
in-place + bisect: 2 | kỳ vọng 2


### 7.3. Cách 4 — Cửa sổ trượt đếm bit: `#(≥ k) − #(≥ k+1)`

`atleast(t)` = số mảng con có AND `≥ t`. Duy trì cửa sổ `[l, r]` với bộ đếm số phần tử có bit `b` bật; AND của cửa sổ = các bit có đếm bằng độ dài cửa sổ. Khi AND `< t`, tăng `l`.

In [6]:
def sliding(nums, k):
    B = 31
    def atleast(t):
        cnt = [0] * B; l = 0; res = 0
        for r, x in enumerate(nums):
            for b in range(B):
                if x >> b & 1:
                    cnt[b] += 1
            while l <= r:
                L = r - l + 1; v = 0
                for b in range(B):
                    if cnt[b] == L:
                        v |= 1 << b
                if v >= t:
                    break
                y = nums[l]
                for b in range(B):
                    if y >> b & 1:
                        cnt[b] -= 1
                l += 1
            res += r - l + 1
        return res
    return atleast(k) - atleast(k + 1)

for a, k, exp in examples:
    print('sliding:', sliding(a, k), '| kỳ vọng', exp)

sliding: 6 | kỳ vọng 6
sliding: 3 | kỳ vọng 3
sliding: 2 | kỳ vọng 2


Đúng nhưng hằng số lớn trong Python (`31` bit × mỗi bước) — dùng để minh hoạ kỹ thuật "đếm đúng bằng hiệu hai lần đếm ít nhất".

### 7.4. Cách 5 — Sparse table + tìm nhị phân biên

`AND` có tính **idempotent** nên sparse table trả lời `AND(l..r)` trong `O(1)`. Với mỗi `r`, tìm nhị phân `l` nhỏ nhất có `AND ≥ k` và `l` nhỏ nhất có `AND ≥ k + 1` (cả hai đơn điệu) ⇒ `O(n log n)`.

In [7]:
def sparse_table(nums, k):
    n = len(nums); sp = [list(nums)]
    j = 1
    while (1 << j) <= n:
        prev = sp[-1]; h = 1 << (j - 1)
        sp.append([prev[i] & prev[i + h] for i in range(n - (1 << j) + 1)])
        j += 1
    def q(l, r):
        t = (r - l + 1).bit_length() - 1
        return sp[t][l] & sp[t][r - (1 << t) + 1]
    def first_l(r, t):                      # l nhỏ nhất với AND(l..r) >= t, hoặc r+1
        lo, hi = 0, r + 1
        while lo < hi:
            mid = (lo + hi) // 2
            if q(mid, r) >= t:
                hi = mid
            else:
                lo = mid + 1
        return lo
    ans = 0
    for r in range(n):
        ans += first_l(r, k + 1) - first_l(r, k)
    return ans

for a, k, exp in examples:
    print('sparse table:', sparse_table(a, k), '| kỳ vọng', exp)

sparse table: 6 | kỳ vọng 6
sparse table: 3 | kỳ vọng 3
sparse table: 2 | kỳ vọng 2


## 8. TRACE từng bước

Trace cách 3 trên `nums = [7, 5, 6, 4, 5, 4]`, `k = 4`: in mảng `b` (= `AND(i..r)`) sau mỗi bước.

In [8]:
def trace(nums, k):
    b = list(nums); ans = 0
    for r, x in enumerate(b):
        i = r - 1; touched = 0
        while i >= 0 and b[i] & x != b[i]:
            b[i] &= x; i -= 1; touched += 1
        add = bisect_right(b, k, 0, r + 1) - bisect_left(b, k, 0, r + 1)
        ans += add
        print(f'r = {r} (x = {x}): ghi đè {touched} ô | b[0..r] = {b[:r + 1]} | số i có AND = {k}: {add} | tổng {ans}')
    print('đáp án:', ans, '| brute:', brute(nums, k))

trace([7, 5, 6, 4, 5, 4], 4)

r = 0 (x = 7): ghi đè 0 ô | b[0..r] = [7] | số i có AND = 4: 0 | tổng 0
r = 1 (x = 5): ghi đè 1 ô | b[0..r] = [5, 5] | số i có AND = 4: 0 | tổng 0
r = 2 (x = 6): ghi đè 2 ô | b[0..r] = [4, 4, 6] | số i có AND = 4: 2 | tổng 2
r = 3 (x = 4): ghi đè 1 ô | b[0..r] = [4, 4, 4, 4] | số i có AND = 4: 4 | tổng 6
r = 4 (x = 5): ghi đè 0 ô | b[0..r] = [4, 4, 4, 4, 5] | số i có AND = 4: 4 | tổng 10
r = 5 (x = 4): ghi đè 1 ô | b[0..r] = [4, 4, 4, 4, 4, 4] | số i có AND = 4: 6 | tổng 16
đáp án: 16 | brute: 16


## 9. Code nộp LeetCode

In [9]:
class Solution:
    def countSubarrays(self, nums: List[int], k: int) -> int:
        b = list(nums)
        ans = 0
        for r, x in enumerate(b):
            i = r - 1
            while i >= 0 and b[i] & x != b[i]:
                b[i] &= x
                i -= 1
            ans += bisect_right(b, k, 0, r + 1) - bisect_left(b, k, 0, r + 1)
        return ans

for a, k, exp in examples:
    got = Solution().countSubarrays(a, k)
    print(got, 'OK' if got == exp else 'SAI')

6 OK
3 OK
2 OK


## 10. Vì sao thuật toán đúng

1. **Bất biến:** trước bước `r`, `b[i] = AND(i..r−1)` cho `i < r`. Cần `b[i] ← b[i] & x`. Nếu tại `i` ta có `b[i] & x = b[i]` (tức `b[i] ⊆ x`), thì vì `b[i'] ⊆ b[i]` với mọi `i' < i` nên `b[i'] ⊆ x` ⇒ `b[i'] & x = b[i']` — không cần cập nhật. Vậy dừng sớm vẫn giữ bất biến.
2. **Đơn điệu:** `b[i] = AND(i..r) ⊆ AND(i+1..r) = b[i+1]` ⇒ `b[i] ≤ b[i+1]` ⇒ `b[0..r]` không giảm, `bisect` hợp lệ.
3. **Khấu hao:** mỗi lần ghi đè `b[i]` mất ít nhất một bit ⇒ mỗi ô bị ghi đè `≤ 30` lần; tổng `O(30 n)` cộng `O(n log n)` cho bisect.
4. Mỗi mảng con `[i, r]` được đếm đúng một lần ở bước `r`.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Brute | `O(n²)` | `O(1)` |
| 2. Từ điển AND | `O(n log V)` | `O(log V)` |
| 3. Ghi đè + bisect | `O(n log V + n log n)` | `O(n)` |
| 4. Cửa sổ trượt bit | `O(n log V)` (hằng số lớn) | `O(log V)` |
| 5. Sparse table | `O(n log n)` | `O(n log n)` |

## 12. Bộ test

### 12.1. Ví dụ đề + ca biên

In [10]:
impls = {'dict': and_dict, 'inplace': inplace_bisect, 'sliding': sliding, 'sparse': sparse_table,
         'submit': lambda a, k: Solution().countSubarrays(a, k)}
edge = [([0], 0, 1), ([5], 3, 0), ([0, 0, 0], 0, 6), ([3, 1, 2], 0, 2), ([2, 3, 2], 2, 5),
        ([10 ** 9], 10 ** 9, 1), ([1, 2, 4, 8], 0, 6)]
for a, k, exp in examples + edge:
    assert brute(a, k) == exp, (a, k, brute(a, k))
    for name, f in impls.items():
        assert f(a, k) == exp, (name, a, k)
print('Ví dụ đề và ca biên: tất cả đúng.')

Ví dụ đề và ca biên: tất cả đúng.


Giải thích: `[3, 1, 2]`, `k = 0`: chỉ `[1, 2]` và `[3, 1, 2]` có AND 0 ⇒ `2`. `[2, 3, 2]`, `k = 2`: mọi mảng con trừ `[3]` ⇒ `5`. `[1, 2, 4, 8]`, `k = 0`: mọi mảng con độ dài `≥ 2` ⇒ `3 + 2 + 1 = 6`.

### 12.2. Đối chiếu ngẫu nhiên với brute

In [11]:
cnt = 0
for _ in range(400):
    n = rng.randint(1, 25)
    mask = rng.choice([1, 3, 7, 15, 255, 2 ** 30 - 1])
    a = [rng.randint(0, mask) for _ in range(n)]
    k = rng.choice([rng.randint(0, mask), a[rng.randint(0, n - 1)], 0])
    exp = brute(a, k)
    for name, f in impls.items():
        assert f(a, k) == exp, (name, a, k)
    cnt += 1
print('Khớp brute trên', cnt, 'bộ ngẫu nhiên.')

Khớp brute trên 400 bộ ngẫu nhiên.


### 12.3. Ca lớn `n = 10^5`

In [12]:
N = 10 ** 5
big = {
    'toàn bằng k': ([123456789] * N, 123456789),
    'ngẫu nhiên': ([rng.randint(0, 10 ** 9) for _ in range(N)], 0),
    'nhiều bit': ([(2 ** 30 - 1) ^ (1 << rng.randint(0, 29)) for _ in range(N)], 0),
}
for name, (a, k) in big.items():
    t = time.time(); x = Solution().countSubarrays(a, k); t1 = time.time() - t
    t = time.time(); y = and_dict(a, k); t2 = time.time() - t
    assert x == y
    print(f'{name:12s}: {x:12d} | nộp {t1:.2f}s | dict {t2:.2f}s')
assert Solution().countSubarrays([123456789] * N, 123456789) == N * (N + 1) // 2
print('Toàn bằng k: n(n+1)/2 mảng con: đúng.')

toàn bằng k :   5000050000 | nộp 0.05s | dict 0.03s


ngẫu nhiên  :   4999528352 | nộp 0.09s | dict 0.07s


nhiều bit   :   4988094913 | nộp 0.35s | dict 0.43s
Toàn bằng k: n(n+1)/2 mảng con: đúng.


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Kiểm `AND & k == k` (chứa đủ bit của `k`) thay vì `AND == k`

In [13]:
def wrong_superset(nums, k):
    ans = 0; cur = {}
    for x in nums:
        nxt = {x: 1}
        for v, c in cur.items():
            nxt[v & x] = nxt.get(v & x, 0) + c
        cur = nxt
        ans += sum(c for v, c in cur.items() if v & k == k)            # SAI
    return ans

print('sai :', wrong_superset(*examples[2][:2]), '| đúng:', Solution().countSubarrays(*examples[2][:2]))

sai : 3 | đúng: 2


**Vì sao sai:** `[3]` có AND `3 ⊇ 2` nhưng `3 ≠ 2`. Muốn dùng "chứa bit" thì phải trừ đi các mảng con có thêm bit thừa.

### Lỗi 2 — Cửa sổ trượt chỉ đếm `#(AND ≥ k)`

In [14]:
def wrong_atleast_only(nums, k):
    n = len(nums); res = 0
    for r in range(n):
        cur = -1
        for l in range(r, -1, -1):
            cur &= nums[l]
            if cur < k:
                break
            res += 1                                                    # SAI: đếm cả AND > k
    return res

print('sai :', wrong_atleast_only(*examples[1][:2]), '| đúng:', Solution().countSubarrays(*examples[1][:2]))

sai : 4 | đúng: 3


**Vì sao sai:** `[2]` có AND `2 ≥ 1` cũng bị đếm. Phải là `#(≥ k) − #(≥ k + 1)`.

### Lỗi 3 — Brute "dừng ngay khi gặp AND = k"

In [15]:
def wrong_break_on_hit(nums, k):
    n = len(nums); cnt = 0
    for l in range(n):
        cur = -1
        for r in range(l, n):
            cur &= nums[r]
            if cur == k:
                cnt += 1
                break                                                    # SAI: các r lớn hơn vẫn có thể = k
    return cnt

print('sai :', wrong_break_on_hit(*examples[0][:2]), '| đúng:', Solution().countSubarrays(*examples[0][:2]))

sai : 3 | đúng: 6


**Vì sao sai:** AND có thể **giữ nguyên** `k` khi kéo dài (ví dụ toàn `1`). Chỉ được dừng khi AND đã **mất** một bit của `k`.

### Lỗi 4 — Từ điển AND quên mảng con một phần tử `[x]`

In [16]:
def wrong_no_singleton(nums, k):
    ans = 0; cur = {}
    for i, x in enumerate(nums):
        nxt = {} if i else {x: 1}                                       # SAI: chỉ thêm [x] ở vị trí đầu
        for v, c in cur.items():
            nxt[v & x] = nxt.get(v & x, 0) + c
        cur = nxt
        ans += cur.get(k, 0)
    return ans

print('sai :', wrong_no_singleton(*examples[0][:2]), '| đúng:', Solution().countSubarrays(*examples[0][:2]))

sai : 3 | đúng: 6


**Vì sao sai:** mỗi `r` phải mở thêm một đầu trái mới `l = r`. Thiếu nó, ta chỉ đếm các mảng con bắt đầu từ 0.

## 14. Mẹo làm bài nhanh & nhận dạng pattern

- Mảng con + AND/OR/GCD ⇒ "số giá trị khác nhau kết thúc tại `r` là `O(log V)`".
- Đếm "đúng bằng" từ đếm "ít nhất": `#(= k) = #(≥ k) − #(≥ k+1)` khi điều kiện đơn điệu.
- Mẹo ghi đè tại chỗ (LC 1521, 3097, 3171) cho code ngắn và nhanh trong Python.

## 15. Biến thể của bài

1. **LC 1521:** AND gần `target` nhất ⇒ duyệt tập giá trị AND.
2. **LC 3171:** OR gần `k` nhất — đối ngẫu (OR chỉ bật thêm bit).
3. Đếm mảng con có **OR = k**: cùng kỹ thuật với `|`.
4. Đếm mảng con có **GCD = k**: số giá trị GCD kết thúc tại `r` cũng `O(log V)`.

In [17]:
# Biến thể 3 và 4: OR = k, GCD = k bằng cùng khung "từ điển giá trị kết thúc tại r"
from math import gcd
def count_op(nums, k, op):
    ans = 0; cur = {}
    for x in nums:
        nxt = {x: 1}
        for v, c in cur.items():
            w = op(v, x); nxt[w] = nxt.get(w, 0) + c
        cur = nxt
        ans += cur.get(k, 0)
    return ans

def brute_op(nums, k, op):
    n = len(nums); cnt = 0
    for l in range(n):
        cur = nums[l]
        for r in range(l, n):
            if r > l:
                cur = op(cur, nums[r])
            cnt += cur == k
    return cnt

for _ in range(200):
    n = rng.randint(1, 15); a = [rng.randint(1, 16) for _ in range(n)]; k = rng.choice(a + [1, 2, 15])
    for op in (lambda x, y: x | y, gcd):
        assert count_op(a, k, op) == brute_op(a, k, op)
print('Biến thể OR = k và GCD = k: khớp brute.')

Biến thể OR = k và GCD = k: khớp brute.


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Ghi chú |
|---|---|---|---|
| 1 | Brute | `O(n²)` | Oracle |
| 2 | Từ điển AND | `O(n log V)` | Tổng quát (AND/OR/GCD) |
| 3 | Ghi đè + bisect | `O(n log V)` | **Nộp bài**, gọn |
| 4 | Cửa sổ trượt bit | `O(n log V)` | Kỹ thuật "≥k − ≥k+1" |
| 5 | Sparse table + nhị phân | `O(n log n)` | Trả lời AND đoạn `O(1)` |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output):** `nums = [4, 6, 7]`, `k = 4`.

**Bài 2 (tính tay):** `nums = [12, 10, 8, 14]`, `k = 8` — liệt kê các mảng con.

**Bài 3 (điền TODO):** trả về **mảng con dài nhất** có AND = `k` (cặp `(l, r)`), hoặc `None`.

**Bài 4 (tự viết lại):** với mỗi `r`, đếm số `l` sao cho `AND(l..r) = k`; trả về mảng `cnt[r]` (tổng của nó là đáp án).

**Bài 5 (mini challenge):** trả lời `q` truy vấn `k` khác nhau trên cùng một mảng (offline): `O(n log V + q)`.

In [18]:
def longest_and_k_todo(nums, k):
    # TODO
    raise NotImplementedError

def per_right_todo(nums, k):
    # TODO
    raise NotImplementedError

def many_queries_todo(nums, ks):
    # TODO
    raise NotImplementedError

print('Đã định nghĩa các hàm TODO (không gọi).')

Đã định nghĩa các hàm TODO (không gọi).


## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) `4 = 100₂, 6 = 110₂, 7 = 111₂`. (2) Mảng con chứa `4` đều AND = 4: `[4], [4,6], [4,6,7]`. (3) `3`.

**Bài 2.** (1) `12 = 1100, 10 = 1010, 8 = 1000, 14 = 1110`. (2) Mảng con một phần tử: chỉ `[8]`. Độ dài ≥ 2: `12 & 10 = 8` nên mọi mảng con chứa cả `12, 10` đều AND = 8; `10 & 8 = 8`, `8 & 14 = 8`; riêng `[10]`, `[12]`, `[14]` không. (3) `[12,10], [12,10,8], [12,10,8,14], [10,8], [10,8,14], [8], [8,14]` ⇒ `7`.

**Bài 3.** (1) Với mỗi `r`, các `l` có AND = `k` là một đoạn `[L, R]` trong `b`. (2) `l` nhỏ nhất là `bisect_left`. (3) Cập nhật độ dài `r − l + 1`.

**Bài 4.** (1) Như lời giải nộp, lưu `add` mỗi bước. (2) `cnt[r] = bisect_right − bisect_left`. (3) Kiểm `sum(cnt) = đáp án`.

**Bài 5.** (1) Mọi mảng con có AND thuộc tập giá trị xuất hiện trong từ điển. (2) Cộng dồn `total[v] += c` cho mọi `(v, c)` ở mỗi bước. (3) Mỗi truy vấn tra `total.get(k, 0)`.

## 19. Đáp án tham khảo cho bài tập

In [19]:
# Bài 1, 2
assert Solution().countSubarrays([4, 6, 7], 4) == 3
ans2 = Solution().countSubarrays([12, 10, 8, 14], 8)
lst = [(l, r) for l in range(4) for r in range(l, 4)
       if __import__('functools').reduce(lambda x, y: x & y, [12, 10, 8, 14][l:r + 1]) == 8]
assert ans2 == len(lst) == 7
print('Bài 1: 3 | Bài 2:', ans2, '| các mảng con (l, r):', lst)

Bài 1: 3 | Bài 2: 7 | các mảng con (l, r): [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 2), (2, 3)]


In [20]:
# Bài 3, 4, 5
def longest_and_k(nums, k):
    b = list(nums); best = None
    for r, x in enumerate(b):
        i = r - 1
        while i >= 0 and b[i] & x != b[i]:
            b[i] &= x; i -= 1
        lo = bisect_left(b, k, 0, r + 1)
        if lo <= r and b[lo] == k and (best is None or r - lo > best[1] - best[0]):
            best = (lo, r)
    return best

def per_right(nums, k):
    b = list(nums); res = []
    for r, x in enumerate(b):
        i = r - 1
        while i >= 0 and b[i] & x != b[i]:
            b[i] &= x; i -= 1
        res.append(bisect_right(b, k, 0, r + 1) - bisect_left(b, k, 0, r + 1))
    return res

def many_queries(nums, ks):
    total = {}; cur = {}
    for x in nums:
        nxt = {x: 1}
        for v, c in cur.items():
            nxt[v & x] = nxt.get(v & x, 0) + c
        cur = nxt
        for v, c in cur.items():
            total[v] = total.get(v, 0) + c
    return [total.get(k, 0) for k in ks]

for _ in range(300):
    n = rng.randint(1, 14); a = [rng.randint(0, 15) for _ in range(n)]; k = rng.choice(a + [0, 5])
    from functools import reduce
    subs = [(l, r) for l in range(n) for r in range(l, n) if reduce(lambda x, y: x & y, a[l:r + 1]) == k]
    lg = longest_and_k(a, k)
    assert (lg is None) == (not subs)
    if subs:
        assert lg[1] - lg[0] == max(r - l for l, r in subs)
    pr = per_right(a, k)
    assert pr == [sum(1 for l, r in subs if r == rr) for rr in range(n)]
    ks = list(range(16))
    assert many_queries(a, ks) == [brute(a, q) for q in ks]
print('Bài 3 (dài nhất), Bài 4 (đếm theo r), Bài 5 (nhiều truy vấn) khớp brute.')

Bài 3 (dài nhất), Bài 4 (đếm theo r), Bài 5 (nhiều truy vấn) khớp brute.


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
b = list(nums); ans = 0
for r, x in enumerate(b):
    i = r - 1
    while i >= 0 and b[i] & x != b[i]:
        b[i] &= x; i -= 1
    ans += bisect_right(b, k, 0, r + 1) - bisect_left(b, k, 0, r + 1)
```

**Checklist:**

- [ ] So sánh `AND == k` (không phải "chứa bit của k" hay "≥ k")?
- [ ] Mỗi `r` mở thêm mảng con `[r, r]`?
- [ ] Giới hạn `bisect` trong `[0, r + 1)` (phần sau chưa cập nhật, không sắp xếp)?
- [ ] Điều kiện dừng ghi đè là `b[i] & x == b[i]`?